# Public Events Transformation to Hourly Detail Features

This notebook converts a manually curated Excel file of Toronto public events into a structured **hourly event-detail dataset** that can be consumed by downstream forecasting jobs.

The goal is to transform event-level information such as:
- event schedule
- attendance
- category
- location
- accessibility / free-event flags

into an **hourly temporal representation** aligned with the forecasting pipeline.

This output is especially important for the multi-hour prediction workflow, where event activity acts as a major external demand driver.

## Process Overview

This notebook performs the following steps:

### 1. Read the source Excel file
A manually curated public-events Excel file is copied from the workspace into the project’s raw storage area and then loaded using a robust binary-to-pandas approach.

### 2. Standardize the schema
Column names are cleaned and normalized so the dataset can be processed consistently even if the original Excel column formatting varies.

### 3. Clean and normalize event fields
The notebook:
- converts latitude and longitude into numeric values
- estimates event attendance
- derives free-event indicators
- creates canonical event identifiers and default values where needed

### 4. Build event timestamps
Start and end timestamps are constructed from:
- event start date
- event end date
- start/end time

Fallback rules are applied when time formats are incomplete.

### 5. Apply data quality filters
Invalid or incomplete records are removed based on:
- missing timestamps
- missing coordinates
- invalid durations
- unrealistic event length

### 6. Generate event-level flags
Additional operational flags are created, including:
- downtown bounding-box indicator
- large-event indicator
- daily start/end bounds

### 7. Expand each event into hourly detail records
Each event is exploded into one record per active hour so the forecasting pipeline can evaluate event activity at hourly granularity.

### 8. Write the hourly detail dataset
The final output is written as a partitioned Parquet dataset:
- `public_events_hourly_detail`

This output is later used by downstream forecasting jobs to derive event-related features.

In [0]:
# ============================================================
# PUBLIC EVENTS – Excel to Hourly DETAIL Features
# Output for Job D (station-event haversine done later in Job D)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType
from io import BytesIO
import pandas as pd

# ------------------------------------------------------------
# 0) PATHS
# ------------------------------------------------------------

SRC_XLSX = "dbfs:/Workspace/Projects/Capstone/data/toronto_events_march_2026.xlsx"
RAW_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw"
RAW_XLSX = f"{RAW_DIR}/toronto_events_march_2026.xlsx"

OUTPUT_PARQUET = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/"
    "public_events_hourly_detail"
)

# ------------------------------------------------------------
# 1) PARAMETERS
# ------------------------------------------------------------

DOWNTOWN_LAT_MIN = 43.640
DOWNTOWN_LAT_MAX = 43.670
DOWNTOWN_LON_MIN = -79.410
DOWNTOWN_LON_MAX = -79.370

MAX_EVENT_HOURS = 72

CAL_START = "2026-03-01 00:00:00"
CAL_END   = "2026-03-31 23:00:00"

# ------------------------------------------------------------
# 2) COPY ONLY THE EXCEL YOU NEED
# ------------------------------------------------------------

dbutils.fs.mkdirs(RAW_DIR)

print("Copying Excel from Workspace to Volume/raw ...")
dbutils.fs.cp(SRC_XLSX, RAW_XLSX, True)

print("Copied file:")
display(dbutils.fs.ls(RAW_DIR))

# ------------------------------------------------------------
# 3) READ EXCEL ROBUSTLY (binaryFile -> BytesIO -> pandas)
# ------------------------------------------------------------

bin_df = spark.read.format("binaryFile").load(RAW_XLSX)
row = bin_df.select("path", "content").first()

print("Binary path:", row["path"])
print("Binary size (bytes):", len(row["content"]))

pdf = pd.read_excel(BytesIO(row["content"]), engine="openpyxl")

print("Pandas shape:", pdf.shape)
print("Pandas dtypes before fix:")
print(pdf.dtypes)
print(pdf.head())

# ------------------------------------------------------------
# 4) FORCE ALL PANDAS COLUMNS TO STRING
# ------------------------------------------------------------

pdf = pdf.fillna("").astype(str)

print("Pandas dtypes after full string cast:")
print(pdf.dtypes)
print(pdf.head())

records = pdf.to_dict("records")
df_raw = spark.createDataFrame(records)

print("Spark raw rows:", df_raw.count())
display(df_raw.limit(10))

# ------------------------------------------------------------
# 5) STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------

def clean_col(c: str) -> str:
    return (
        c.strip()
         .lower()
         .replace(" ", "_")
         .replace("/", "_")
         .replace("-", "_")
         .replace("(", "")
         .replace(")", "")
         .replace(".", "")
         .replace("__", "_")
    )

df = df_raw
for old in df.columns:
    df = df.withColumnRenamed(old, clean_col(old))

print("Columns after standardization:")
print(df.columns)

# ------------------------------------------------------------
# 6) NORMALIZE EXPECTED COLUMN NAMES
# ------------------------------------------------------------

rename_candidates = {
    "mean_attendance_day": ["mean_attendance_day", "mean_attendance__day"],
    "event_start_date": ["event_start_date"],
    "event_end_date": ["event_end_date"],
    "event_time_starts": ["event_time_starts"],
    "event_time_ends": ["event_time_ends"],
    "event_attendance": ["event_attendance"],
    "event_category": ["event_category"],
    "free_event": ["free_event"],
    "lat": ["lat"],
    "lon": ["lon"],
    "event_name": ["event_name"]
}

def first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

resolved = {}
for target, candidates in rename_candidates.items():
    resolved[target] = first_existing(df.columns, candidates)

required = ["event_start_date", "event_end_date", "event_time_starts", "event_time_ends", "lat", "lon"]
missing = [c for c in required if resolved[c] is None]

if missing:
    raise Exception(f"Missing required columns after cleaning: {missing}. Found columns: {df.columns}")

for canonical, found in resolved.items():
    if found is not None and found != canonical:
        df = df.withColumnRenamed(found, canonical)

print("Columns after canonical normalization:")
print(df.columns)

# ------------------------------------------------------------
# 7) BASIC CLEANING
# ------------------------------------------------------------

if "event_id" not in df.columns:
    df = df.withColumn("event_id", F.monotonically_increasing_id())

if "event_name" not in df.columns:
    df = df.withColumn("event_name", F.lit("unknown_event"))

if "event_category" not in df.columns:
    df = df.withColumn("event_category", F.lit("unknown"))

df = (
    df
    .withColumn("event_lat", F.col("lat").cast(DoubleType()))
    .withColumn("event_lon", F.col("lon").cast(DoubleType()))
)

if "mean_attendance_day" in df.columns:
    df = df.withColumn(
        "attendance_est",
        F.regexp_replace(F.col("mean_attendance_day").cast(StringType()), "[^0-9.]", "").cast(DoubleType())
    )
elif "event_attendance" in df.columns:
    df = df.withColumn(
        "attendance_est",
        F.regexp_replace(F.col("event_attendance").cast(StringType()), "[^0-9.]", "").cast(DoubleType())
    )
else:
    df = df.withColumn("attendance_est", F.lit(0.0))

df = df.fillna({"attendance_est": 0.0})

if "free_event" in df.columns:
    df = df.withColumn(
        "free_event_flag",
        F.when(F.lower(F.col("free_event").cast(StringType())).isin("yes", "y", "true", "1"), 1).otherwise(0)
    )
else:
    df = df.withColumn("free_event_flag", F.lit(0))

# ------------------------------------------------------------
# 8) BUILD start_ts / end_ts
# ------------------------------------------------------------

df = (
    df
    .withColumn("event_start_date_dt", F.to_date(F.col("event_start_date"), "yyyy-MM-dd"))
    .withColumn("event_end_date_dt", F.to_date(F.col("event_end_date"), "yyyy-MM-dd"))
)

df = (
    df
    .withColumn(
        "start_ts",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.date_format(F.col("event_start_date_dt"), "yyyy-MM-dd"),
                F.col("event_time_starts").cast(StringType())
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "end_ts",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.date_format(F.col("event_end_date_dt"), "yyyy-MM-dd"),
                F.col("event_time_ends").cast(StringType())
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
)

# fallback if hour is HH:mm
df = df.withColumn(
    "start_ts",
    F.coalesce(
        F.col("start_ts"),
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.date_format(F.col("event_start_date_dt"), "yyyy-MM-dd"),
                F.col("event_time_starts").cast(StringType())
            ),
            "yyyy-MM-dd HH:mm"
        )
    )
)

df = df.withColumn(
    "end_ts",
    F.coalesce(
        F.col("end_ts"),
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.date_format(F.col("event_end_date_dt"), "yyyy-MM-dd"),
                F.col("event_time_ends").cast(StringType())
            ),
            "yyyy-MM-dd HH:mm"
        )
    )
)

df = df.withColumn(
    "end_ts",
    F.coalesce(F.col("end_ts"), F.expr("start_ts + interval 2 hours"))
)

# ------------------------------------------------------------
# 9) DATA QUALITY FILTERS
# ------------------------------------------------------------

df = df.filter(
    F.col("start_ts").isNotNull() &
    F.col("end_ts").isNotNull() &
    F.col("event_lat").isNotNull() &
    F.col("event_lon").isNotNull() &
    (F.col("end_ts") >= F.col("start_ts"))
)

df = (
    df
    .withColumn("duration_hours", (F.unix_timestamp("end_ts") - F.unix_timestamp("start_ts")) / 3600.0)
    .filter(F.col("duration_hours") >= 0)
    .filter(F.col("duration_hours") <= F.lit(MAX_EVENT_HOURS))
)

print("Rows after DQ:", df.count())

display(
    df.select(
        "event_id", "event_name", "start_ts", "end_ts", "duration_hours",
        "event_lat", "event_lon", "attendance_est", "event_category", "free_event_flag"
    ).limit(10)
)

# ------------------------------------------------------------
# 10) EVENT-LEVEL FLAGS
# ------------------------------------------------------------

df = df.withColumn(
    "is_downtown_bbox",
    F.when(
        (F.col("event_lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
        (F.col("event_lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX)),
        1
    ).otherwise(0)
)

df = df.withColumn(
    "big_event_flag",
    F.when(F.col("attendance_est") >= 5000, 1).otherwise(0)
)

# Also keep day bounds for Job D day-level derivations
df = (
    df
    .withColumn("start_date", F.to_date("start_ts"))
    .withColumn("end_date", F.to_date("end_ts"))
)

# ------------------------------------------------------------
# 11) EXPAND EVENT -> HOURLY DETAIL
# ------------------------------------------------------------

df_h = (
    df
    .withColumn("start_h", F.date_trunc("hour", F.col("start_ts")))
    .withColumn("end_h", F.date_trunc("hour", F.col("end_ts")))
    .withColumn(
        "hour_seq",
        F.sequence(F.col("start_h"), F.col("end_h"), F.expr("interval 1 hour"))
    )
    .withColumn("event_hour_ts", F.explode("hour_seq"))
    .filter(
        (F.col("event_hour_ts") >= F.to_timestamp(F.lit(CAL_START))) &
        (F.col("event_hour_ts") <= F.to_timestamp(F.lit(CAL_END)))
    )
    .withColumn("year", F.year("event_hour_ts").cast(IntegerType()))
    .withColumn("month", F.month("event_hour_ts").cast(IntegerType()))
    .withColumn("day", F.dayofmonth("event_hour_ts").cast(IntegerType()))
    .withColumn("hour", F.hour("event_hour_ts").cast(IntegerType()))
)

print("Expanded hourly detail rows:", df_h.count())

display(
    df_h.select(
        "event_id", "event_name", "start_ts", "end_ts", "event_hour_ts",
        "attendance_est", "event_lat", "event_lon", "is_downtown_bbox", "big_event_flag"
    ).limit(20)
)

# ------------------------------------------------------------
# 12) BUILD HOURLY DETAIL OUTPUT
# ------------------------------------------------------------

events_hourly_detail = (
    df_h
    .select(
        "year",
        "month",
        "day",
        "hour",
        F.col("event_hour_ts").alias("target_ts_hour"),
        "event_id",
        "event_name",
        "event_category",
        "start_ts",
        "end_ts",
        "start_date",
        "end_date",
        "event_lat",
        "event_lon",
        "attendance_est",
        "is_downtown_bbox",
        "big_event_flag",
        "free_event_flag"
    )
    .orderBy("year", "month", "day", "hour", "event_id")
)

print("Hourly detail rows final:", events_hourly_detail.count())
display(events_hourly_detail.limit(30))

# ------------------------------------------------------------
# 13) OPTIONAL VALIDATIONS
# ------------------------------------------------------------

display(
    events_hourly_detail.groupBy("year", "month")
    .count()
    .orderBy("year", "month")
)

display(
    events_hourly_detail.select(
        "attendance_est",
        "event_lat",
        "event_lon"
    ).summary("count", "min", "25%", "50%", "75%", "max")
)

display(
    events_hourly_detail.groupBy("year", "month", "day", "hour")
    .agg(
        F.countDistinct("event_id").alias("num_active_events"),
        F.sum("attendance_est").alias("sum_attendance_active_events")
    )
    .orderBy("year", "month", "day", "hour")
    .limit(48)
)

# ------------------------------------------------------------
# 14) WRITE OUTPUT
# ------------------------------------------------------------

(
    events_hourly_detail
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(OUTPUT_PARQUET)
)

print("Written to:", OUTPUT_PARQUET)

display(dbutils.fs.ls(OUTPUT_PARQUET))

## Outputs

This notebook produces the following output dataset:

Path: `public_events_hourly_detail`

### Output Characteristics
The resulting dataset contains one row per:

- event
- active forecast hour

Each row includes:
- event timestamp hour
- event identifier and name
- category
- attendance estimate
- geographic location
- hourly activity alignment
- operational event flags

### Main Output Fields
Examples of the generated fields include:
- `target_ts_hour`
- `event_id`
- `event_name`
- `event_category`
- `attendance_est`
- `event_lat`
- `event_lon`
- `is_downtown_bbox`
- `big_event_flag`
- `free_event_flag`

This dataset is used later by:
- the 1-hour prediction workflow
- the multi-hour forecasting workflow
- downstream event-feature derivation logic

## Key Insights and Summary

### 1. Events were converted from static records into operational hourly signals
The source Excel file contains event-level information, but forecasting requires hourly alignment. This notebook bridges that gap by transforming each event into an hourly activity timeline.

---

### 2. Event timing is critical for demand forecasting
By expanding events hour by hour, the system can later estimate which stations may be affected during specific time windows rather than only knowing that an event exists on a given day.

---

### 3. Attendance and location improve demand relevance
The inclusion of:
- attendance estimates
- geographic coordinates
- large-event indicators
- downtown location flags

allows downstream jobs to quantify the potential operational impact of different public events.

---

### 4. The dataset is flexible and reusable
Although this example uses a manually prepared Excel file for March 2026, the notebook is designed to process any future Toronto public events file that follows a similar structure.

This makes the approach reusable beyond a single month.

---

### 5. Business relevance
Public events can create sudden and localized changes in bike-sharing demand. This notebook ensures that the forecasting system can incorporate event-driven demand shifts into its prediction logic.

In practical terms, this notebook provides the **event intelligence layer** used to enrich future-hour forecasting with real-world urban activity context.